In [3]:
import numpy as np
import pygame
import pickle
import os
import time
from typing import Tuple, List, Dict

# ==================== QLearningFighter 类 ====================
class QLearningFighter:
    """Q学习格斗智能体"""
    ACTIONS: List[str] = ["闪避", "格挡", "招架", "轻击", "破防", "重击"]
    ACTION_PROPS: Dict[int, Tuple[int, int]] = {
        0: (2, 1), 1: (3, 0), 2: (2, 2), 3: (2, 1), 4: (2, 1), 5: (1, 2)
    }
    HP_RANGE = [1, 2, 3, 4, 5]
    STA_RANGE = [1, 2, 3, 4, 5]
    SPD_RANGE = [1, 2, 3]
    SPD_TAGS = ["自己快", "对方慢", "无"]

    def __init__(self,
                 q_table_path="q_table.pkl",
                 alpha=0.2,
                 gamma=0.95,
                 epsilon=1.0,
                 epsilon_decay=0.0025,
                 epsilon_min=0.015,
                 max_turns=50,
                 win_reward=100,
                 lose_penalty=-80,
                 small_reward=2.0,
                 small_penalty=-2.0,
                 defense_no_attack_penalty=-0.5):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.max_turns = max_turns
        self.win_reward = win_reward
        self.lose_penalty = lose_penalty
        self.small_reward = small_reward
        self.small_penalty = small_penalty
        self.defense_no_attack_penalty = defense_no_attack_penalty

        self.q_table_path = q_table_path
        self.q_table: Dict = {}
        self.train_rewards: List[float] = []
        self._init_q_table()

    def _init_q_table(self):
        if os.path.exists(self.q_table_path):
            with open(self.q_table_path, 'rb') as f:
                self.q_table = pickle.load(f)
            print(f"Loaded Q-table: {self.q_table_path}")
        else:
            for s_hp in self.HP_RANGE:
                for s_sta in self.STA_RANGE:
                    for s_spd in self.SPD_RANGE:
                        for o_hp in self.HP_RANGE:
                            for o_sta in self.STA_RANGE:
                                for o_spd in self.SPD_RANGE:
                                    for tag in self.SPD_TAGS:
                                        self.q_table[(s_hp, s_sta, s_spd, o_hp, o_sta, o_spd, tag)] = [0.0]*6

    def save_q_table(self):
        with open(self.q_table_path, 'wb') as f:
            pickle.dump(self.q_table, f)
        print(f"Saved Q-table: {self.q_table_path}")

    def select_action(self, state: Tuple, is_agent=True) -> int:
        if is_agent:
            if np.random.rand() < self.epsilon:
                return np.random.choice(6)
            return np.argmax(self.q_table[state])
        else:
            o_hp, o_sta, o_spd = state
            probs = np.array([0.15, 0.15, 0.15, 0.2, 0.2, 0.15])
            if o_hp <= 2:
                probs[[0,1,2]] *= 1.8
                probs[[3,4,5]] *= 0.5
            if o_spd == 3:
                probs[5] *= 2.0
            if o_sta <= 2:
                probs[[2,5]] *= 0.1
            probs /= probs.sum()
            return np.random.choice(6, p=probs)

    def fight_step(self, state: Tuple, a_act: int, o_act: int) -> Tuple[float, Tuple, bool]:
        """核心对抗逻辑"""
        s_hp, s_sta, s_spd, o_hp, o_sta, o_spd, tag = state
        reward = 0.0
        done = False

        # 防御无攻击惩罚
        defense_actions = [0,1,2]
        attack_actions = [3,4,5]
        if (a_act in defense_actions) and (o_act not in attack_actions) and (self.ACTION_PROPS[a_act][1] > 0):
            reward += self.defense_no_attack_penalty

        # 基础属性计算
        s_consume = self.ACTION_PROPS[a_act][1]
        o_consume = self.ACTION_PROPS[o_act][1]
        s_sta_new = max(1, s_sta - s_consume)
        o_sta_new = max(1, o_sta - o_consume)
        s_exhausted = (s_sta - s_consume) <= 0
        o_exhausted = (o_sta - o_consume) <= 0
        s_final_spd = s_spd + self.ACTION_PROPS[a_act][0]
        o_final_spd = o_spd + self.ACTION_PROPS[o_act][0]
        s_hp_temp, o_hp_temp = s_hp, o_hp

        # 对抗规则
        if a_act == o_act and s_final_spd == o_final_spd:
            if a_act == 3:  # 双方轻击
                base = 1
                s_hp_temp -= base + (1 if s_exhausted else 0)
                o_hp_temp -= base + (1 if o_exhausted else 0)
            elif a_act == 5:  # 双方重击
                base = 2
                s_hp_temp -= base + (1 if s_exhausted else 0)
                o_hp_temp -= base + (1 if o_exhausted else 0)
        else:
            # 智能体攻击奖励
            if a_act == 5:  # 重击
                if o_act in [3,4] and o_final_spd >= s_final_spd:
                    reward += self.small_penalty
                elif o_act == 1:
                    base = 1
                    o_hp_temp -= base + (1 if o_exhausted else 0)
                    reward += self.small_reward
                elif s_final_spd >= o_final_spd + 1:
                    base = 2
                    o_hp_temp -= base + (1 if o_exhausted else 0)
                    reward += self.small_reward * 2
            elif a_act == 2 and o_act in [3,5] and s_final_spd >= o_final_spd:
                base = 1
                o_hp_temp -= base + (1 if o_exhausted else 0)
                reward += self.small_reward * 2
            elif a_act == 0 and o_act in [3,5] and s_final_spd >= o_final_spd:
                reward += self.small_reward * 1.5
            elif a_act == 4:  # 破防
                if (o_act in [2,0,5] and s_final_spd >= o_final_spd) or (o_act == 3 and s_final_spd > o_final_spd):
                    reward += self.small_reward * 2

            # 敌方攻击惩罚
            if o_act == 3 and o_final_spd > s_final_spd:
                base = 1
                s_hp_temp -= base + (1 if s_exhausted else 0)
                reward += self.small_penalty
            elif o_act == 5 and o_final_spd >= s_final_spd + 1:
                base = 2
                s_hp_temp -= base + (1 if s_exhausted else 0)
                reward += self.small_penalty * 2
            elif o_act == 2 and a_act in [3,5] and o_final_spd >= s_final_spd:
                base = 1
                s_hp_temp -= base + (1 if s_exhausted else 0)
                reward += self.small_penalty

            # 格挡重击特殊
            if a_act == 1 and o_act == 5:
                s_sta_new = max(1, s_sta_new - 2)
                base = 1
                s_hp_temp -= base + (1 if (s_sta - s_consume - 2) <= 0 else 0)
                reward += self.small_penalty * 2
            if o_act == 1 and a_act == 5:
                o_sta_new = max(1, o_sta_new - 2)
                base = 1
                o_hp_temp -= base + (1 if (o_sta - o_consume - 2) <= 0 else 0)
                reward += self.small_reward * 2

        # 体力回复
        s_sta_new = min(5, s_sta_new + 1)
        o_sta_new = min(5, o_sta_new + 1)

        # 胜负判定
        s_hp_final = max(0, s_hp_temp)
        o_hp_final = max(0, o_hp_temp)
        if s_hp_final <= 0:
            reward += self.lose_penalty
            done = True
        if o_hp_final <= 0:
            reward += self.win_reward
            done = True

        # 速度标签
        next_tag = "无"
        if a_act == 4:
            if (o_act in [2,0,5] and s_final_spd >= o_final_spd) or (o_act == 3 and s_final_spd > o_final_spd):
                next_tag = "自己快"
        elif a_act == 2 and o_act in [3,5] and s_final_spd >= o_final_spd:
            next_tag = "对方慢"
        elif o_act == 2 and a_act in [3,5] and o_final_spd >= s_final_spd:
            next_tag = "对方慢"
        elif a_act == 5 and s_final_spd >= o_final_spd + 1 and o_act != 1:
            next_tag = "自己快"

        if next_tag == "自己快":
            s_spd_next = 3
            o_spd_next = np.random.choice(self.SPD_RANGE)
        elif next_tag == "对方慢":
            s_spd_next = np.random.choice(self.SPD_RANGE)
            o_spd_next = 1
        else:
            s_spd_next = np.random.choice(self.SPD_RANGE)
            o_spd_next = np.random.choice(self.SPD_RANGE)

        next_state = (
            int(s_hp_final), int(s_sta_new), int(s_spd_next),
            int(o_hp_final), int(o_sta_new), int(o_spd_next),
            next_tag
        )
        next_state = next_state if next_state in self.q_table else state
        return reward, next_state, done


# ==================== Pygame 界面 ====================
def draw_bar(surf, x, y, w, h, val, maxv, bg, fg):
    pygame.draw.rect(surf, bg, (x, y, w, h))
    fill = int(w * val / maxv)
    if fill > 0:
        pygame.draw.rect(surf, fg, (x, y, fill, h))

def draw_text(surf, text, font, x, y, color, center=False):
    img = font.render(text, True, color)
    if center:
        rect = img.get_rect(center=(x, y))
        surf.blit(img, rect)
    else:
        surf.blit(img, (x, y))

def draw_stickman(surf, x, y, action, side, color=(0,0,0), dashed=False):
    """
    绘制火柴人，根据英文动作名调整姿态。
    side: 'left' 或 'right'
    dashed: 闪避残影
    """
    # 基础点（腰部中心为(x,y)）
    head = (x, y-35)
    neck = (x, y-25)
    waist = (x, y+15)
    shoulder_l = (x-12, y-22)
    shoulder_r = (x+12, y-22)
    hip_l = (x-10, y+10)
    hip_r = (x+10, y+10)

    # 默认手脚位置（自然下垂）
    hand_l = (x-15, y-5)
    hand_r = (x+15, y-5)
    foot_l = (x-12, y+60)
    foot_r = (x+12, y+60)

    # 根据动作调整
    if action == 'Light':      # 轻击
        if side == 'left':
            hand_r = (x+35, y-15)
        else:
            hand_l = (x-35, y-15)
    elif action == 'Heavy':    # 重击
        hand_l = (x-30, y-10)
        hand_r = (x+30, y-10)
    elif action == 'Break':    # 破防（踢腿）
        if side == 'left':
            foot_r = (x+25, y+65)
        else:
            foot_l = (x-25, y+65)
    elif action == 'Block':    # 格挡
        if side == 'left':
            hand_r = (x+10, y-20)
        else:
            hand_l = (x-10, y-20)
    elif action == 'Parry':    # 招架
        hand_l = (x-10, y-20)
        hand_r = (x+10, y-20)
    # Dodge（闪避）无特殊武器动作

    # 线条颜色（残影用浅灰）
    line_color = (180,180,180) if dashed else color
    line_width = 3

    def draw_line(p1, p2):
        pygame.draw.line(surf, line_color, p1, p2, line_width)

    # 绘制身体
    draw_line(head, neck)
    draw_line(neck, waist)
    draw_line(shoulder_l, hand_l)
    draw_line(shoulder_r, hand_r)
    draw_line(hip_l, foot_l)
    draw_line(hip_r, foot_r)
    # 头部圆圈
    pygame.draw.circle(surf, line_color, head, 8, 2 if dashed else 0)

    # 绘制剑（仅攻击/格挡/招架动作）
    if action in ['Light', 'Heavy', 'Block', 'Parry']:
        if action == 'Light':
            if side == 'left':
                sword_start = hand_r
                sword_end = (hand_r[0] + 35, hand_r[1] - 10)
            else:
                sword_start = hand_l
                sword_end = (hand_l[0] - 35, hand_l[1] - 10)
        elif action == 'Heavy':
            sword_start = (x, y-15)
            sword_end = (x, y-65)
        elif action == 'Block':
            if side == 'left':
                sword_start = (x+15, y-25)
                sword_end = (x-10, y-15)
            else:
                sword_start = (x-15, y-25)
                sword_end = (x+10, y-15)
        else:  # Parry
            sword_start = (x+5, y-25)
            sword_end = (x-5, y-15)
        # 剑用亮黄色粗线
        pygame.draw.line(surf, (255,255,0), sword_start, sword_end, 5)

def show_menu(screen, font_large, font_med):
    screen.fill((200,200,200))
    draw_text(screen, "Q-learning Agent vs Rule-based Opponent", font_large, 450, 250, (0,0,0), center=True)
    draw_text(screen, "Press any key to start", font_med, 450, 350, (0,0,0), center=True)
    pygame.display.flip()
    waiting = True
    while waiting:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                exit()
            if event.type == pygame.KEYDOWN:
                waiting = False

def run_demo(fighter: QLearningFighter):
    pygame.init()
    screen = pygame.display.set_mode((900, 650))
    pygame.display.set_caption("Q-learning Fighting Demo")
    clock = pygame.time.Clock()

    # 颜色
    BLACK = (0,0,0)
    RED = (220,20,60)
    GREEN = (50,205,50)
    BLUE = (30,144,255)
    GRAY = (128,128,128)
    LIGHT = (200,200,200)
    DARK_RED = (139,0,0)
    DARK_GREEN = (0,100,0)
    YELLOW = (255,215,0)

    # 字体
    try:
        font_large = pygame.font.SysFont('Arial', 32)
        font_med = pygame.font.SysFont('Arial', 24)
        font_small = pygame.font.SysFont('Arial', 20)
    except:
        font_large = pygame.font.Font(None, 32)
        font_med = pygame.font.Font(None, 24)
        font_small = pygame.font.Font(None, 20)

    show_menu(screen, font_large, font_med)

    # 英文动作映射
    action_en = ["Dodge", "Block", "Parry", "Light", "Break", "Heavy"]
    tag_en = {"自己快": "Self Fast", "对方慢": "Opp Slow", "无": "None"}

    def reset():
        s_spd = np.random.choice(fighter.SPD_RANGE)
        o_spd = np.random.choice(fighter.SPD_RANGE)
        return (5,5,s_spd,5,5,o_spd,"无"), 0.0, 0, False

    state, total_score, turn, done = reset()
    last_score = 0.0
    agent_act_name = "Waiting"
    opp_act_name = "Waiting"
    running = True
    pause = 0.8
    last_step = time.time()
    fighter.epsilon = 0.0  # 完全利用

    while running:
        now = time.time()
        for e in pygame.event.get():
            if e.type == pygame.QUIT:
                running = False
            elif e.type == pygame.KEYDOWN:
                if e.key == pygame.K_r:
                    state, total_score, turn, done = reset()
                    last_score = 0.0
                    agent_act_name = "Waiting"
                    opp_act_name = "Waiting"
                elif e.key == pygame.K_q:
                    running = False

        if not done and (now - last_step) >= pause:
            agent_act = np.argmax(fighter.q_table[state])
            opp_act = fighter.select_action((state[3], state[4], state[5]), is_agent=False)
            r, next_state, done = fighter.fight_step(state, agent_act, opp_act)
            total_score += r
            turn += 1
            last_score = r
            agent_act_name = action_en[agent_act]
            opp_act_name = action_en[opp_act]
            state = next_state
            last_step = now

        # 绘制
        screen.fill(LIGHT)
        s_hp, s_sta, s_spd, o_hp, o_sta, o_spd, tag = state

        draw_text(screen, "Q-learning Fighting", font_large, 450, 30, BLACK, center=True)
        draw_text(screen, f"Turn: {turn}", font_med, 100, 70, BLACK)
        draw_text(screen, f"Total Score: {total_score:.1f}", font_med, 300, 70, BLACK)
        tag_show = tag_en.get(tag, tag)
        draw_text(screen, f"Tag: {tag_show}", font_med, 600, 70, DARK_RED)

        # 左侧智能体
        draw_text(screen, "Agent", font_med, 150, 120, BLUE, center=True)
        draw_bar(screen, 50, 150, 200, 20, s_hp, 5, GRAY, RED)
        draw_text(screen, f"HP:{s_hp}/5", font_small, 160, 150, BLACK, center=True)
        draw_bar(screen, 50, 180, 200, 15, s_sta, 5, GRAY, GREEN)
        draw_text(screen, f"Stamina:{s_sta}/5", font_small, 160, 180, BLACK, center=True)
        draw_text(screen, f"Speed:{s_spd}", font_med, 160, 210, BLACK, center=True)
        draw_text(screen, f"Action:{agent_act_name}", font_med, 160, 240, DARK_GREEN, center=True)

        # 右侧对手
        draw_text(screen, "Opponent", font_med, 750, 120, RED, center=True)
        draw_bar(screen, 650, 150, 200, 20, o_hp, 5, GRAY, RED)
        draw_text(screen, f"HP:{o_hp}/5", font_small, 750, 150, BLACK, center=True)
        draw_bar(screen, 650, 180, 200, 15, o_sta, 5, GRAY, GREEN)
        draw_text(screen, f"Stamina:{o_sta}/5", font_small, 750, 180, BLACK, center=True)
        draw_text(screen, f"Speed:{o_spd}", font_med, 750, 210, BLACK, center=True)
        draw_text(screen, f"Action:{opp_act_name}", font_med, 750, 240, DARK_RED, center=True)

        # 本回合得分
        scolor = YELLOW if last_score >= 0 else RED
        draw_text(screen, f"Turn Score: {last_score:+.1f}", font_large, 450, 280, scolor, center=True)

        # 火柴人
        if agent_act_name == 'Dodge':
            draw_stickman(screen, 150, 400, agent_act_name, 'left', dashed=True)
            draw_stickman(screen, 130, 400, agent_act_name, 'left', color=BLACK)
        else:
            draw_stickman(screen, 150, 400, agent_act_name, 'left', color=BLACK)

        if opp_act_name == 'Dodge':
            draw_stickman(screen, 750, 400, opp_act_name, 'right', dashed=True)
            draw_stickman(screen, 770, 400, opp_act_name, 'right', color=BLACK)
        else:
            draw_stickman(screen, 750, 400, opp_act_name, 'right', color=BLACK)

        if done:
            result = "You Win!" if o_hp <= 0 else "You Lose!" if s_hp <= 0 else "Game Over"
            draw_text(screen, result, font_large, 450, 500, DARK_RED, center=True)
            draw_text(screen, "Press R to restart | Q to quit", font_med, 450, 550, BLACK, center=True)

        draw_text(screen, "Auto battle", font_small, 450, 620, GRAY, center=True)
        pygame.display.flip()
        clock.tick(30)

    pygame.quit()


if __name__ == "__main__":
    q_table_path = "q_table.pkl"  # 使用相对路径
    fighter = QLearningFighter(
        q_table_path=q_table_path,
        alpha=0.2,
        gamma=0.95,
        epsilon=1.0,
        epsilon_decay=0.0025,
        epsilon_min=0.015,
        max_turns=50,
        win_reward=100,
        lose_penalty=-80,
        small_reward=2.0,
        small_penalty=-2.0,
        defense_no_attack_penalty=-0.5
    )
    run_demo(fighter)

Loaded Q-table: q_table.pkl
